# BYOV (Bring Your Own Variable)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/your_own_variable.ipynb)

This notebook demonstrates how to create a custom derived variable by subclassing `DerivedVariable`. Uses a dewpoint depression example. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import xarray as xr
import extremeweatherbench as ewb
from extremeweatherbench.derived import DerivedVariable
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2017 Lucifer European Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9005,
    title="2017 Lucifer European Heat Wave (demo)",
    start_date=datetime.datetime(2017, 8, 2),
    end_date=datetime.datetime(2017, 8, 5),
    location=BoundingBoxRegion.create_region(
        latitude_min=39.0,
        latitude_max=45.0,
        longitude_min=12.0,
        longitude_max=18.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
class DewpointDepression(DerivedVariable):
    """2 m dewpoint depression (T2m - Td2m) in Kelvin."""

    variables = [
        "surface_air_temperature",
        "surface_dewpoint_temperature",
    ]

    def __init__(self, name: str = "dewpoint_depression"):
        super().__init__(name=name)

    def derive_variable(
        self, data: xr.Dataset, *args, **kwargs
    ) -> xr.DataArray:
        depression = (
            data["surface_air_temperature"]
            - data["surface_dewpoint_temperature"]
        )
        depression.name = self.name
        return depression

In [ ]:
dd = DewpointDepression()

forecast = ewb.ZarrForecast(
    source="gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr",
    name="HRES",
    variable_mapping=ewb.HRES_metadata_variable_mapping,
    storage_options={"remote_options": {"anon": True}},
)

target = ewb.ERA5(
    variables=[
        "surface_air_temperature",
        "surface_dewpoint_temperature",
    ]
)

eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=[
            ewb.metrics.MeanAbsoluteError(
                forecast_variable=dd,
                target_variable=dd,
            ),
        ],
        target=target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()
print(outputs)